In [47]:
import time
import datetime
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.remote.webelement import WebElement
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import requests
from bs4 import BeautifulSoup
from itertools import chain

In [ ]:
# How many times to scroll the page and get a batch of articles
max_num_scrols = 10

In [ ]:
def get_pages(num_pages: int):
    for i in range(num_pages):
        response = requests.get(
            f"https://www.ufc.com/trending/all?page={i}", headers={
            "User-Agent": "Mozilla/5.0 (Windows NT 11.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/134.0.6998.166 Safari/537.36"
        })

In [16]:
responses = [
    requests.get(f"https://www.ufc.com/trending/all?page={page}", headers={
    "User-Agent": "Mozilla/5.0 (Windows NT 11.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/134.0.6998.166 Safari/537.36"
    })
    for page in range(max_num_scrols)
]
responses

[<Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>,
 <Response [200]>]

In [21]:
pages = [r.text for r in responses]

In [28]:
first_page = pages[0]
s1 = BeautifulSoup(first_page, "html.parser")
ls1 = s1.find_all("a", "c-card--grid-card-trending")
l1 = ls1[0]
l1

<a class="c-card--grid-card-trending grid_card_image" data-aos="fade-up" data-locale="en" data-modal-callback="/ajax/modal/node/150984" data-video-meta-id="858764" data-video-meta-type="vod" href="/video/150984">
<div class="c-card--grid-card-trending__4-wrapper">
<div class="c-card--grid-card-trending__image">
<div class="layout layout--onecol">
<div class="layout__region layout__region--content">
<picture>
<source height="324" media="(min-width: 1024px)" srcset="https://ufc.com/images/styles/homepage_grid_big_item_desktop_x1/s3/2025-09/091625-ciryl-gane-paris-interview.jpg?itok=xnFLaxq1 1x, https://ufc.com/images/styles/homepage_grid_big_item_desktop_x2/s3/2025-09/091625-ciryl-gane-paris-interview.jpg?itok=m7RPsiau 2x" type="image/jpeg" width="576"/>
<source height="411" media="(min-width: 700px)" srcset="/s3/files/styles/homepage_grid_tablet_x1/s3/2025-09/091625-ciryl-gane-paris-interview.jpg?itok=c16CYhGD 1x, /s3/files/styles/homepage_grid_tablet_x2/s3/2025-09/091625-ciryl-gane-par

In [30]:
[d for d in l1.descendants]

['\n',
 <div class="c-card--grid-card-trending__4-wrapper">
 <div class="c-card--grid-card-trending__image">
 <div class="layout layout--onecol">
 <div class="layout__region layout__region--content">
 <picture>
 <source height="324" media="(min-width: 1024px)" srcset="https://ufc.com/images/styles/homepage_grid_big_item_desktop_x1/s3/2025-09/091625-ciryl-gane-paris-interview.jpg?itok=xnFLaxq1 1x, https://ufc.com/images/styles/homepage_grid_big_item_desktop_x2/s3/2025-09/091625-ciryl-gane-paris-interview.jpg?itok=m7RPsiau 2x" type="image/jpeg" width="576"/>
 <source height="411" media="(min-width: 700px)" srcset="/s3/files/styles/homepage_grid_tablet_x1/s3/2025-09/091625-ciryl-gane-paris-interview.jpg?itok=c16CYhGD 1x, /s3/files/styles/homepage_grid_tablet_x2/s3/2025-09/091625-ciryl-gane-paris-interview.jpg?itok=UYoTUapM 2x" type="image/jpeg" width="730"/>
 <source height="200" srcset="https://ufc.com/images/styles/homepage_grid_mobile_x1/s3/2025-09/091625-ciryl-gane-paris-interview.jpg

In [36]:
article_type = l1.find_all("span", "c-card--grid-card-trending__info-prefix")[0]
article_type.text

'Interviews'

In [35]:
time_ago = l1.find_all("div", "field--name-datetime")[0]
time_ago.text

'1 hour ago'

In [39]:
headline = l1.find_all("h3", "c-card--grid-card-trending__headline")[0]
headline.text.strip()

'Ciryl Gane Pre-Fight Interview | UFC 321'

In [40]:
link = l1["href"]
link

'/video/150984'

In [49]:
def get_articles_data(page_content: str):
    soup = BeautifulSoup(page_content, "html.parser")
    articles = soup.find_all("a", "c-card--grid-card-trending")

    articles_data = [
        {
            "type": link.find_all("span", "c-card--grid-card-trending__info-prefix")[0].text,
            "time_ago": link.find_all("div", "field--name-datetime")[0].text,
            "headline": link.find_all("h3", "c-card--grid-card-trending__headline")[0].text.strip(),
            "link": link["href"]
        }
        for link in articles
    ]

    for data in articles_data:
        yield data

In [52]:
flattened_articles = list(chain.from_iterable(get_articles_data(page) for page in pages))
flattened_articles

[{'type': 'Interviews',
  'time_ago': '1 hour ago',
  'headline': 'Ciryl Gane Pre-Fight Interview | UFC 321',
  'link': '/video/150984'},
 {'type': 'Interviews',
  'time_ago': '1 hour ago',
  'headline': 'Tom Aspinall Pre-Fight Interview | UFC 321',
  'link': '/video/150983'},
 {'type': "Dana White's Contender Series",
  'time_ago': '3 hours ago',
  'headline': "Week 6 Preview | Dana White's Contender Series…",
  'link': '/news/week-6-preview-dana-whites-contender-series-season-9'},
 {'type': 'Announcements',
  'time_ago': '20 hours ago',
  'headline': 'Updates To UFC Vancouver',
  'link': '/news/updates-ufc-vancouver'},
 {'type': 'Highlights',
  'time_ago': '20 hours ago',
  'headline': 'Greatest Finishes | Noche UFC: Silva vs Lopes',
  'link': '/video/150976'},
 {'type': 'Fight Coverage',
  'time_ago': '20 hours ago',
  'headline': 'The Scorecard | Noche UFC',
  'link': '/news/scorecard-noche-ufc-san-antonio'},
 {'type': 'Athletes',
  'time_ago': '21 hours ago',
  'headline': 'KYLE D

In [ ]:
for page in pages:
    soup = BeautifulSoup(page, "html.parser")
    articles = soup.find_all("a", "c-card--grid-card-trending")

    articles_data = [
        {
            "type": link.find_all("span", "c-card--grid-card-trending__info-prefix")[0].text,
            "time_ago": link.find_all("div", "field--name-datetime")[0].text,
            "headline": link.find_all("h3", "c-card--grid-card-trending__headline")[0].text.strip(),
            "link": link["href"]
        }
        for link in articles
    ]


[{'type': 'Interviews', 'time_ago': '1 hour ago', 'headline': 'Ciryl Gane Pre-Fight Interview | UFC 321', 'link': '/video/150984'}, {'type': 'Interviews', 'time_ago': '1 hour ago', 'headline': 'Tom Aspinall Pre-Fight Interview | UFC 321', 'link': '/video/150983'}, {'type': "Dana White's Contender Series", 'time_ago': '3 hours ago', 'headline': "Week 6 Preview | Dana White's Contender Series…", 'link': '/news/week-6-preview-dana-whites-contender-series-season-9'}, {'type': 'Announcements', 'time_ago': '20 hours ago', 'headline': 'Updates To UFC Vancouver', 'link': '/news/updates-ufc-vancouver'}, {'type': 'Highlights', 'time_ago': '20 hours ago', 'headline': 'Greatest Finishes | Noche UFC: Silva vs Lopes', 'link': '/video/150976'}, {'type': 'Fight Coverage', 'time_ago': '20 hours ago', 'headline': 'The Scorecard | Noche UFC', 'link': '/news/scorecard-noche-ufc-san-antonio'}, {'type': 'Athletes', 'time_ago': '21 hours ago', 'headline': 'KYLE DAUKAUS | THE D’ARCE KNIGHT RETURNS', 'link': '

In [ ]:
#c-card--grid-card-trending__headline
